In [ ]:
"""eod_sale_service — data-quality gate. Scoped to the run day (gold accumulates one
replaceWhere partition per report_date). Grain guard: one row per (transaction_id,
product_code) — the grain proven duplicate-free in the source ClickHouse table."""
from pyspark.sql import functions as F


def _service_checks(df, running_date):
    rd = str(running_date)[:10]
    day = df.where(F.col("report_date").cast("string") == F.lit(rd))
    total = day.count()
    a = day.agg(
        F.sum(F.when(F.col("transaction_id").isNull(), 1).otherwise(0)).alias("null_txn"),
        F.sum(F.when(F.col("product_code").isNull(), 1).otherwise(0)).alias("null_prod"),
        F.sum(F.when(F.col("report_date").isNull(), 1).otherwise(0)).alias("null_date"),
        F.sum(F.when(F.col("total_amount").isNull(), 1).otherwise(0)).alias("null_amt"),
        F.sum(F.when(~F.col("service_name").isin("Pay Bill", "Pay Card", "Top Up"), 1)
              .otherwise(0)).alias("bad_service"),
    ).first()
    grain_dups = total - day.select("transaction_id", "product_code").distinct().count()
    return [
        (f"row_count > 0 for {rd}", total > 0, f"rows={total}"),
        ("no null transaction_id", (a["null_txn"] or 0) == 0, f"nulls={a['null_txn']}"),
        ("no null product_code", (a["null_prod"] or 0) == 0, f"nulls={a['null_prod']}"),
        ("no null report_date", (a["null_date"] or 0) == 0, f"nulls={a['null_date']}"),
        ("no null total_amount", (a["null_amt"] or 0) == 0, f"nulls={a['null_amt']}"),
        ("service_name in known set", (a["bad_service"] or 0) == 0, f"unknown={a['bad_service']}"),
        ("grain unique (transaction_id, product_code)", grain_dups == 0, f"dups={grain_dups}"),
    ]


In [ ]:
def run_service_dq(ctx) -> None:
    df = ctx.spark.read.table(ctx.gold("fact_eod_sale_service"))
    results = _service_checks(df, ctx.running_date)
    for name, ok, detail in results:
        ctx.logger.info(f"[DQ] {'PASS' if ok else 'FAIL'} — {name} ({detail})")
    failed = [f"{n} ({d})" for n, ok, d in results if not ok]
    if failed:
        raise ValueError("DQ(service) FAILED: " + " | ".join(failed))
    ctx.logger.info(f"[DQ] all {len(results)} checks passed for {str(ctx.running_date)[:10]}")
